In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError("Install plotly first: pip install plotly") from exc

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

data_path = Path(r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_all_states.csv")
geojson_path = Path(r"C:\Users\hadap\Downloads\DSM_Project\India GeoJSON\state\india_state.geojson")

if not data_path.exists():
    raise FileNotFoundError(f"Could not find data file: {data_path.resolve()}")

if not geojson_path.exists():
    raise FileNotFoundError(f"Could not find GeoJSON file: {geojson_path.resolve()}")

df = pd.read_csv(data_path)

state_name_map = {
    "Andaman & Nicobar Islands": "Andaman and Nicobar",
    "Andaman and Nicobar Islands": "Andaman and Nicobar",
    "NCT of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Odisha": "Orissa",
    "Pondicherry": "Puducherry",
    "Uttarakhand": "Uttaranchal",
}

def normalize_state_name(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    return state_name_map.get(value, value)

map_df = df[[
    "state",
    "overall_priority_score",
    "overall_priority_rank",
    "policy_priority_category",
    "st_literacy_rate_pct",
    "dropout_secondary_pct",
    "st_bpl_mean_pct",
    "employment_wpr_person_per_1000",
    "mgnreg_sought_not_received_per_1000",
]].copy()

map_df["state"] = map_df["state"].map(normalize_state_name)

# Higher priority score means worse distress, so invert it:
# negative/red = higher concern, positive/green = lower concern
score_centered = map_df["overall_priority_score"] - map_df["overall_priority_score"].mean()
scale = np.abs(score_centered).max() or 1
map_df["map_score"] = (-score_centered / scale * 100).round(1)

with geojson_path.open("r", encoding="utf-8") as f:
    india_geojson = json.load(f)

feature_key = "NAME_1"

geojson_states = set()
for feature in india_geojson["features"]:
    properties = feature.get("properties", {})
    properties[feature_key] = normalize_state_name(properties.get(feature_key))
    geojson_states.add(properties[feature_key])

missing_in_geojson = sorted(set(map_df["state"]) - geojson_states)
if missing_in_geojson:
    print("States in CSV but not matched in GeoJSON:", missing_in_geojson)

fig = px.choropleth(
    map_df,
    geojson=india_geojson,
    featureidkey="properties.NAME_1",
    locations="state",
    color="map_score",
    color_continuous_scale=[
        [0.0, "#8b0000"],
        [0.25, "#d73027"],
        [0.5, "#f7f7bf"],
        [0.75, "#66bd63"],
        [1.0, "#1a9850"],
    ],
    range_color=(-100, 100),
    hover_name="state",
    hover_data={
        "map_score": True,
        "overall_priority_rank": True,
        "st_literacy_rate_pct": True,
        "dropout_secondary_pct": True,
        "st_bpl_mean_pct": True,
        "employment_wpr_person_per_1000": True,
        "mgnreg_sought_not_received_per_1000": True,
        "policy_priority_category": True,
    },
    title="India State Priority Map",
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(marker_line_color="white", marker_line_width=0.8)
fig.update_layout(
    width=950,
    height=700,
    margin={"r": 10, "t": 60, "l": 10, "b": 10},
    coloraxis_colorbar_title="Score",
)

details_columns = [
    "state",
    "map_score",
    "overall_priority_rank",
    "policy_priority_category",
    "st_literacy_rate_pct",
    "dropout_secondary_pct",
    "st_bpl_mean_pct",
    "employment_wpr_person_per_1000",
    "mgnreg_sought_not_received_per_1000",
]

if HAS_WIDGETS:
    fig_widget = go.FigureWidget(fig)
    details = widgets.HTML()

    def render_state_details(state_name):
        row = map_df.loc[map_df["state"] == state_name, details_columns].iloc[0]
        details.value = f"""
        <div style='padding:12px 14px;border:1px solid #d9d9d9;border-radius:10px;background:#fafafa; min-width:320px;'>
          <h3 style='margin:0 0 8px 0;'>{row['state']}</h3>
          <p style='margin:4px 0;'><b>Map score:</b> {row['map_score']}</p>
          <p style='margin:4px 0;'><b>Priority rank:</b> {row['overall_priority_rank']}</p>
          <p style='margin:4px 0;'><b>Category:</b> {row['policy_priority_category']}</p>
          <p style='margin:4px 0;'><b>ST literacy:</b> {row['st_literacy_rate_pct']}%</p>
          <p style='margin:4px 0;'><b>Secondary dropout:</b> {row['dropout_secondary_pct']}%</p>
          <p style='margin:4px 0;'><b>ST BPL mean:</b> {row['st_bpl_mean_pct']}%</p>
          <p style='margin:4px 0;'><b>WPR (per 1000):</b> {row['employment_wpr_person_per_1000']}</p>
          <p style='margin:4px 0;'><b>MGNREG unmet demand (per 1000):</b> {row['mgnreg_sought_not_received_per_1000']}</p>
        </div>
        """

    def handle_click(trace, points, selector):
        if points.point_inds:
            state_name = trace.locations[points.point_inds[0]]
            render_state_details(state_name)

    fig_widget.data[0].on_click(handle_click)
    render_state_details(map_df.sort_values("overall_priority_rank").iloc[0]["state"])
    display(widgets.HBox([fig_widget, details]))
else:
    fig.show()
    print("For clickable side-panel details, install ipywidgets: pip install ipywidgets")
    display(map_df[details_columns].sort_values("overall_priority_rank").head(10))


States in CSV but not matched in GeoJSON: ['Dadra and Nagar Haveli and Daman and Diu']


c:\Users\hadap\AppData\Local\Programs\Python\Python312\Lib\site-packages\jupyter_client\session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



    'data': [{'coloraxis': 'coloraxis',
              'customdata': array([[-81.…